## 1. 테스트 환경 세팅

### 필요한 패키지
```bash
pip install pytest pytest-cov httpx freezegun
```
FastAPI는 `TestClient`를 쓰려면 내부적으로 `httpx`가 필요합니다.

### 프로젝트 구조 (실무에서 흔한 패턴)
```
app/
  main.py
  services/
    auth.py
    signup.py
tests/
  conftest.py          # 공통 fixture 모음
  test_auth.py         # 단위 테스트
  test_api_signup.py   # API 통합 테스트
```

### pytest 설정 파일
```ini
# pytest.ini
[pytest]
testpaths = tests
python_files = test_*.py
```

---

## 2. 테스트의 종류 3단계 (반드시 구분해서 이해하기)

| 종류 | 대상 | 속도 | 예시 |
|---|---|---|---|
| **단위 테스트** | 함수 하나 | 매우 빠름 | 비밀번호 해싱 함수 |
| **통합 테스트** | 여러 컴포넌트 연결 | 보통 | API 엔드포인트 전체 흐름 |
| **E2E 테스트** | 실제 서비스 전체 | 느림 | 브라우저로 회원가입 끝까지 |

지금 단계에서는 **단위 테스트 + 통합 테스트** 위주로 충분합니다. E2E는 나중에 여유 생기면 봐도 됩니다.

---

## 3. Fixture — 테스트용 DB 세션 준비 (conftest.py)

실제 운영 DB를 테스트에서 건드리면 절대 안 되니, 테스트 전용 DB(보통 SQLite in-memory)를 씁니다.

```python
# tests/conftest.py
import pytest
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker
from app.database import Base
from app.main import app, get_db
from fastapi.testclient import TestClient

TEST_DB_URL = "sqlite:///:memory:"
engine = create_engine(TEST_DB_URL, connect_args={"check_same_thread": False})
TestingSessionLocal = sessionmaker(bind=engine)

@pytest.fixture
def db_session():
    Base.metadata.create_all(bind=engine)   # 테스트마다 테이블 새로 생성
    session = TestingSessionLocal()
    yield session
    session.close()
    Base.metadata.drop_all(bind=engine)     # 끝나면 정리

@pytest.fixture
def client(db_session):
    def override_get_db():
        yield db_session
    app.dependency_overrides[get_db] = override_get_db  # 실제 DB 대신 테스트 DB 주입
    yield TestClient(app)
    app.dependency_overrides.clear()
```

이 fixture 하나만 만들어두면 이후 모든 테스트에서 `client`, `db_session`을 매개변수로 받기만 하면 자동으로 깨끗한 테스트 DB가 준비됩니다.

---

## 4. 단위 테스트 — 순수 함수부터 시작

가장 쉬운 것부터: 외부 의존성 없는 함수를 테스트합니다.

```python
# tests/test_auth.py
from app.services.auth import hash_password, verify_password

def test_hash_password_creates_different_hash_each_time():
    """salt 때문에 같은 비밀번호도 매번 다른 해시가 나와야 함"""
    assert hash_password("pass1234") != hash_password("pass1234")

def test_verify_correct_password_returns_true():
    hashed = hash_password("pass1234")
    assert verify_password("pass1234", hashed) is True

def test_verify_wrong_password_returns_false():
    hashed = hash_password("pass1234")
    assert verify_password("wrongpass", hashed) is False
```

### 경계값 테스트 (실무에서 놓치기 쉬운 부분)
```python
def test_empty_password_raises_error():
    with pytest.raises(ValueError):
        hash_password("")

def test_password_exactly_min_length_is_valid():
    """정확히 8자일 때 (경계값) 통과해야 함"""
    result = validate_password_length("12345678")
    assert result is True

def test_password_one_char_short_is_invalid():
    """7자일 때 (경계값 바로 아래) 실패해야 함"""
    result = validate_password_length("1234567")
    assert result is False
```

**경계값 테스트가 왜 중요한가**: `< 8`인지 `<= 8`인지 부호 하나 잘못 쓰는 실수는 실무에서 정말 흔합니다. 정확히 경계 지점(7자, 8자, 9자)을 테스트해두면 이런 실수를 바로 잡아냅니다.

---

## 5. 시간 관련 테스트 — 인증번호 만료 (freezegun)

```python
# tests/test_verification.py
from freezegun import freeze_time
from datetime import datetime
from app.services.verification import is_code_expired

@freeze_time("2026-01-01 12:00:00")
def test_code_valid_just_before_expiry():
    created = datetime.now()
    with freeze_time("2026-01-01 12:04:59"):  # 4분 59초 후
        assert is_code_expired(created, ttl_minutes=5) is False

@freeze_time("2026-01-01 12:00:00")
def test_code_expired_just_after_expiry():
    created = datetime.now()
    with freeze_time("2026-01-01 12:05:01"):  # 5분 1초 후
        assert is_code_expired(created, ttl_minutes=5) is True
```
시간을 실제로 기다리지 않고 "시간을 고정"시켜서 정확히 원하는 시점을 테스트할 수 있습니다. 실무에서 "만료", "쿨다운" 로직 테스트할 때 거의 필수로 씁니다.

---

## 6. Mock — 외부 의존성(이메일 발송) 가짜로 대체

```python
# tests/test_signup.py
from unittest.mock import patch

@patch("app.services.signup.send_verification_email")
def test_signup_sends_email_with_correct_address(mock_send_email, db_session):
    from app.services.signup import signup_user
    signup_user("test@example.com", "pass1234", db_session)

    mock_send_email.assert_called_once()
    args, kwargs = mock_send_email.call_args
    assert args[0] == "test@example.com"

@patch("app.services.signup.send_verification_email")
def test_signup_does_not_call_email_when_email_already_exists(mock_send_email, db_session):
    """이미 존재하는 이메일이면 이메일 발송 자체가 시도되지 않아야 함"""
    from app.services.signup import signup_user
    # 미리 유저 하나 생성
    create_existing_user(db_session, "test@example.com")

    with pytest.raises(EmailAlreadyExistsError):
        signup_user("test@example.com", "pass1234", db_session)

    mock_send_email.assert_not_called()  # 호출 안 됐는지 확인
```

**왜 mock을 쓰나**: 테스트를 100번 돌리는데 실제로 이메일이 100번 발송되면 안 되고, 이메일 서버가 느리거나 다운돼도 테스트가 실패하면 안 됩니다. "이메일 발송 함수가 올바른 인자로 호출됐는지"만 검증하고, 실제 발송 여부는 별도로 신뢰합니다.

---

## 7. API 통합 테스트 — 실제 요청/응답 흐름 검증

```python
# tests/test_api_signup.py

def test_signup_success_returns_201(client):
    response = client.post("/api/v1/auth/signup", json={
        "email": "new@example.com",
        "password": "SecurePass123!"
    })
    assert response.status_code == 201
    body = response.json()
    assert body["success"] is True
    assert "user_id" in body["data"]

def test_signup_with_invalid_email_returns_422(client):
    """Pydantic 검증에서 걸러지는 케이스"""
    response = client.post("/api/v1/auth/signup", json={
        "email": "not-an-email",
        "password": "SecurePass123!"
    })
    assert response.status_code == 422

def test_signup_with_duplicate_email_returns_409(client, db_session):
    client.post("/api/v1/auth/signup", json={"email": "dup@example.com", "password": "pass1234!"})
    response = client.post("/api/v1/auth/signup", json={"email": "dup@example.com", "password": "pass5678!"})

    assert response.status_code == 409
    assert response.json()["error"]["code"] == "EMAIL_ALREADY_EXISTS"

def test_verify_code_with_correct_code_succeeds(client, db_session):
    # given: 회원가입 후 실제 생성된 인증번호를 DB에서 직접 조회
    client.post("/api/v1/auth/signup", json={"email": "verify@example.com", "password": "pass1234!"})
    code = get_verification_code_from_db(db_session, "verify@example.com")

    # when
    response = client.post("/api/v1/auth/verification-codes/verify", json={
        "email": "verify@example.com",
        "code": code
    })

    # then
    assert response.status_code == 200

def test_verify_code_with_wrong_code_returns_400(client, db_session):
    client.post("/api/v1/auth/signup", json={"email": "wrong@example.com", "password": "pass1234!"})

    response = client.post("/api/v1/auth/verification-codes/verify", json={
        "email": "wrong@example.com",
        "code": "000000"
    })
    assert response.status_code == 400
```

**Given-When-Then 패턴**을 주석으로 표시해두면 테스트가 뭘 검증하는지 한눈에 읽힙니다. 실무에서 많이 쓰는 관례입니다.

---

## 8. 테스트 이름 짓는 법 (실무에서 중요하게 봄)

```python
# 나쁜 예 - 뭘 테스트하는지 이름만 보고 모름
def test_signup():
    ...
def test_1():
    ...

# 좋은 예 - "무엇을_어떤조건에서_어떤결과" 패턴
def test_signup_with_duplicate_email_returns_409():
    ...
def test_verify_code_after_expiry_returns_400():
    ...
```
테스트가 실패했을 때 이름만 보고 뭐가 문제인지 바로 알 수 있어야 좋은 테스트 이름입니다.

---

## 9. 커버리지 확인 및 목표 설정

```bash
pytest --cov=app --cov-report=term-missing tests/
```

```
Name                     Stmts   Miss  Cover   Missing
-------------------------------------------------------
app/services/auth.py        20      2    90%   45-46
app/services/signup.py      35      8    77%   60-67
-------------------------------------------------------
TOTAL                       55     10    82%
```
`Missing` 컬럼이 테스트 안 된 줄 번호입니다. 100%를 목표로 하기보다, **핵심 비즈니스 로직(인증, 결제, 권한 체크)은 반드시 커버하고, 단순 getter/setter는 굳이 안 해도 된다**는 감각을 가지시면 됩니다.

---

## 10. 지금 프로젝트에 적용할 순서

1. `conftest.py`에 테스트용 DB fixture부터 만들기
2. 회원가입 로직의 순수 함수(비밀번호 해싱, 인증번호 생성)부터 단위 테스트 작성
3. 이메일 발송 부분을 mock으로 처리해서 signup 서비스 함수 테스트
4. API 엔드포인트 통합 테스트 (성공/중복이메일/잘못된인증번호 케이스별로)
5. 커버리지 돌려보고 빠진 부분(특히 에러 케이스) 채워넣기
